# Time Series Analysis

## Exploring Time Series Data

In [ ]:
import pandas as pd
import numpy as np
import datetime, calendar
from statsmodels.tsa.seasonal import seasonal_decompose,STL
from statsmodels.tsa.statespace.tools import diff
from statsmodels.tsa.stattools import acf
from statsmodels.tsa.forecasting.theta import ThetaModel
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace import exponential_smoothing

from statsmodels.tsa.api import ( 
  ExponentialSmoothing, SimpleExpSmoothing, Holt, STLForecast
) 

import pmdarima as pm

from scipy.cluster import hierarchy
from scipy.spatial.distance import pdist,squareform

from ind5003 import ts

import matplotlib.pyplot as plt
import seaborn as sns

### Example: Basic plot of housing data

In [ ]:
#| fig-align: center
#| fig-cap: Housing sales time plot
#| label: fig-one-fam-houses
#| fig-pos: 'ht'

hsales = pd.read_csv('data/hsales.csv', parse_dates=[0])
hsales.set_index('date', inplace=True)
hsales.index.freq = 'MS'
hsales.plot(title='Sales of One-Family Houses, USA', legend=False, figsize=(12,4))
plt.xlabel('Month-Year'); plt.ylabel('Sales');

In [ ]:
hsales.resample('14D').interpolate().head()

### Example: Season plots of housing data

In [ ]:
#| fig-align: center
#| fig-cap: "Season plot of housing sales"
#| label: fig-season-sales
#| fig-pos: 'ht'

hsales.loc[:, 'year'] = hsales.index.year
hsales.loc[:, 'month'] = hsales.index.month

yrs = np.sort(hsales.year.unique())
color_ids = np.linspace(0, 1, num=len(yrs))
colors_to_use = plt.cm.YlOrRd(color_ids)

plt.figure(figsize=(10, 6))

for i,yr in enumerate(yrs):
    df_tmp = hsales.loc[hsales.year == yr, :]
    plt.plot(df_tmp.month, df_tmp.hsales, color=colors_to_use[i]);
    plt.text(12.1, df_tmp.hsales.iloc[-1], str(yr), color=colors_to_use[i])
plt.title('Season Plot: House Sales')
plt.xlim(0, 13)
plt.xticks(np.arange(1, 13), calendar.month_abbr[1:13]);

### Example: Australian quarterly electricity production

In [ ]:
#| fig-align: center
#| fig-cap: Aus electric production time plot
#| label: fig-aus-elec-time
#| fig-pos: 'ht'

qau = pd.read_csv('data/qauselec.csv', parse_dates=[0])
qau.set_index('date', inplace=True)
qau.index.freq = 'QS'

qau.plot(figsize=(8,4), title='Electrcity Production by Quarter', legend=False)
plt.xlabel('Qtr-Year'); plt.ylabel('Billion kWh');

In [ ]:
#| fig-align: center
#| fig-cap: "Season plot, electric production"
#| label: fig-season-qauselec
#| fig-pos: 'ht'

qau2 = qau.copy()

qau2.loc[:, 'year'] = qau2.index.year
qau2.loc[:, 'qtr'] = qau2.index.quarter
yrs = np.sort(qau2.year.unique())
color_ids = np.linspace(0, 1, num=len(yrs))
colors_to_use = plt.cm.YlOrRd(color_ids)

plt.figure(figsize=(10, 5))

for i,yr in enumerate(yrs):
    df_tmp = qau2.loc[qau2.year == yr, :]
    plt.plot(df_tmp.qtr, df_tmp.kWh, color=colors_to_use[i]);
    plt.text(4.1, df_tmp.kWh.iloc[-1], str(yr), color=colors_to_use[i])
plt.title('Season Plot: Electricity Production')
plt.xlim(0, 5)
plt.xticks(np.arange(1, 5), ['Q1', 'Q2', 'Q3', 'Q4']);

### Example: Lag plots of housing sales data

In [ ]:
#| fig-align: center
#| fig-cap: Lag plots of housing sales data
#| label: fig-lag-sales
#| fig-pos: 'ht'

f, aa = plt.subplots(nrows=3,ncols=4, sharex=True, sharey=True)
f.set_figheight(9)
f.set_figwidth(12)

y = hsales.hsales.values
for i in np.arange(0, 3):
    for j in np.arange(0, 4):
        lag = i*4 + j + 1
        aa[i,j].scatter(y[:-lag], y[lag:], alpha=0.4)
        aa[i,j].set_xlabel("lag " + str(lag))
f.suptitle('Lag plots');

## Decomposing Time Series Data
### Example: Additive decomposition, housing sales

In [ ]:
#| fig-align: center
#| fig-cap: "Additive decomposition"
#| label: fig-add-decomp-hsales
#| fig-pos: 'ht'

hsales_add = seasonal_decompose(hsales.loc[:, 'hsales'], 
                                model='additive', extrapolate_trend='freq')
ax = hsales_add.plot()
ax.set_figheight(5)

### Example: Multiplicative decomposition, Aus electricity data

In [ ]:
#| fig-align: center
#| fig-cap: "Additive decomposition"
#| label: fig-mult-decomp-qau
#| fig-pos: 'ht'

qau_mult = seasonal_decompose(qau.loc[:, 'kWh'], model='multiplicative',
                              extrapolate_trend='freq')
ax = qau_mult.plot();
ax.set_figheight(5)

### Example: STL decomposition, Aus electricity

In [ ]:
#| fig-align: center
#| fig-cap: "STL decomposition, electricity data"
#| label: fig-stl-decomp
#| fig-pos: 'ht'
qau_stl = STL(qau.kWh).fit()
ax = qau_stl.plot();
ax.set_figheight(5)

## Forecasting
### Benchmark methods
### Example: Benchmark forecasts housing sales

In [ ]:
#| fig-align: center
#| fig-cap: "Benchmark forecasts, housing sales data"
#| label: fig-benchmark-hsales
#| fig-pos: 'ht'

# Set aside the last two years as the test set.
#hsales = hsales.drop(columns=['year', 'month'])
train_set = hsales.iloc[:-24,]
test_set = hsales.iloc[-24:, ]

# Obtain the forecast from the training set
mean_forecast = ts.meanf(train_set.hsales, 24)
snaive_forecast = ts.snaive(train_set.hsales, 24, 12)

# Plot the predictions and true values
ax = train_set.hsales.plot(title='Benchmarks', legend=False, figsize=(12,4.5))
test_set.hsales.plot(ax=ax, legend=False, style='--')
mean_forecast.plot(ax=ax, legend=True, style='-')
snaive_forecast.plot(ax=ax, legend=False, style='-')
plt.legend(labels=['train', 'test', 'mean', 'snaive'], loc='lower right');

### Example: Benchmark forecast errors

In [ ]:
for x in [ts.rmse, ts.mae]:
    print(f"{x.__name__},mean: {x(test_set.hsales.values, mean_forecast.values):.3f}")
    print(f"{x.__name__},snaive: {x(test_set.hsales.values, snaive_forecast.values):.3f}")
    print('---')

In [ ]:
ts.mase(test_set.hsales.values,  snaive_forecast.values, train_set.values, 
        seasonality=12)

### ARIMA Models
### Example: Dow Jones index

In [ ]:
#| fig-align: center
#| fig-cap: Dow Jones, time series plot
#| label: fig-dow-ts
#| fig-pos: 'ht'

dj = pd.read_csv('data/dj.csv')
dj.plot(legend=False, title='Dow Jones Index', figsize=(12,4));

In [ ]:
#| fig-align: center
#| fig-cap: Dow Jones, differenced plot
#| label: fig-dow-diff-ts
#| fig-pos: 'ht'

ddj = diff(dj)
ddj.plot(legend=False, title='Differenced Dow Jones', figsize=(12,4));

In [ ]:
#| fig-align: center
#| fig-cap: "ACF, before and after differencing"
#| label: fig-acf-diff
#| fig-pos: 'ht'
 
plt.figure(figsize=(8, 5))
plt.subplot(211)
plt.stem(acf(dj, fft=False))
plt.title("Non-differenced")
plt.subplot(212)
plt.stem(acf(ddj, fft=False));
plt.title("Differenced Series");

### Example: Auto ARIMA on Aus electricity

In [ ]:
train2 = qau.kWh[:-12]
test2 = qau.kWh[-12:]

snaive_f = ts.snaive(train2, 12, 4)
print(f"The RMSE is approximately {ts.rmse(test2.values, snaive_f.values):.3f}.")

In [ ]:
#| echo: false
import warnings
warnings.filterwarnings(
    "ignore",
    message=r".*'force_all_finite' was renamed to 'ensure_all_finite'.*",
    category=FutureWarning,
)
arima_m1 = pm.auto_arima(train2.values, seasonal=True, m=4, test='adf', 
                         trace=False, suppress_warnings=True)

In [ ]:
#| eval: false
arima_m1 = pm.auto_arima(train2.values, seasonal=True, m=4, test='adf', 
                         trace=False, suppress_warnings=True)

In [ ]:
arima_m1.summary()

In [ ]:
#| fig-align: center
#| fig-cap: "ARIMA residual diagnostics"
#| label: fig-arima-diag
#| fig-pos: 'ht'
arima_m1.plot_diagnostics(figsize=(12,6));

In [ ]:
rmse1 = ts.rmse(test2.values, arima_m1.predict(n_periods=12))
mase1 = ts.mase(test2.values, arima_m1.predict(n_periods=12), 
                train2.values, seasonality=4)
print(f"The RMSE on the test set is {rmse1:.3f}.")
print(f"The MASE on the test set is {mase1:.3f}.") 

In [ ]:
#| fig-align: center
#| label: fig-arima-fc
#| fig-cap: "ARIMA forecasts"
#| fig-pos: 'ht'
n_periods = 12
fc, confint = arima_m1.predict(n_periods=n_periods, return_conf_int=True)

ff = pd.Series(fc, index=test2.index)
lower_series = pd.Series(confint[:, 0], index=test2.index)
upper_series = pd.Series(confint[:, 1], index=test2.index)

plt.figure(figsize=(14, 5))
plt.plot(train2[-36:])
plt.plot(ff, color='red', label='Forecast')
plt.fill_between(lower_series.index, lower_series, upper_series, color='gray', alpha=.15)
plt.plot(test2, 'g--', label='True')
plt.legend();

### ETS 
### Example: ETS model on Aus electricity dataset 

In [ ]:
ets_m1 = ExponentialSmoothing(train2.values, seasonal_periods=4, trend='add', 
                              seasonal='add')
ets_fit = ets_m1.fit()
ets_fit.summary()

In [ ]:
#| fig-align: center
#| fig-cap: ETS trend and level components
#| label: fig-ets-components
#| fig-pos: 'ht'

plt.subplot(211)
plt.plot(train2.index, ets_fit.trend, scaley=False)
plt.ylim(0.245, 0.255)
plt.ylabel('slope')

plt.subplot(212)
plt.plot(train2.index, ets_fit.level)
plt.ylabel('level');

In [ ]:
ets_fc = ets_fit.forecast(steps=12)
print(f"The RMSE on the test set is {ts.rmse(test2.values, ets_fc):.3f}.")

### Theta Method
### Example: International tourism arrivals to Singapore

In [ ]:
#| fig-align: center
#| label: fig-sg-tourism-ts
#| fig-cap: Singapore monthly tourist arrivals
#| fig-pos: 'ht'

tourism = pd.read_excel("data/international_visitor_arrivals_sg.xlsx", 
                        parse_dates=[0], date_format="%Y %b", 
                        na_values='na')
sea_tourism = tourism.iloc[:, 0:9]
sea_tourism.columns = ['Date', 'SEA', 'Brunei', 'Indonesia' ,'Malaysia', 
                       'Myanmar', 'Philippines', 'Thailand', 'Vietnam']
sea_tourism.Date = sea_tourism.Date.str.replace('  ', '')
sea_tourism.Date = pd.to_datetime(sea_tourism.Date, format="%Y %b")
sea_tourism.set_index('Date', inplace=True)
sea_tourism.sort_index(inplace=True)
sea_tourism.index.freq = 'MS'

sea2 = sea_tourism.SEA[sea_tourism.index < datetime.datetime(2020, 1, 1)]
fig = sea2.plot(figsize=(12,4));
fig.set_title('Monthly Arrivals from SEA Countries before COVID-19');

In [ ]:
theta_mod = ThetaModel(sea2)
theta_mod_fit = theta_mod.fit()
print(theta_mod_fit.summary())

In [ ]:
#| fig-align: center
#| fig-cap: "Theta decomposition, showing the breakdown into long- and short-term trends"
#| label: fig-theta-decomp
#| fig-pos: 'ht'

# carry out mutliplicative decomposition in order to obtain seasonally adjusted series
sea2_mult = seasonal_decompose(sea2, model='multiplicative', extrapolate_trend='freq')

# get long-term trend slope and intercept
a0 = (sea2.mean() - theta_mod_fit.params['b0']*(503)/2)/1e6
xvals = np.arange(1, 505)
yvals1 = a0 + theta_mod_fit.params['b0']*(xvals - 1)
yvals1 = pd.Series(data=yvals1, index=sea2.index)

# get short term trend series
a2 = -1*a0
b2 = -1*theta_mod_fit.params['b0']
yvals2 = a2 + b2*(xvals - 1) + 2*(sea2_mult.resid + sea2_mult.trend)
yvals2 = pd.Series(data=yvals2, index=sea2.index)
fig = (sea2_mult.resid + sea2_mult.trend).plot(label='Seasonally adjusted', 
                                               legend=True, figsize=(12,4));
yvals1.plot(legend=True, label='Long-term trend', style="--")
yvals2.plot(legend=True, label='Short-term trend', style="--");
fig.set_title("Theta decomposition");

In [ ]:
#| fig-align: center
#| fig-cap: "Theta decomposition forecasts for tourism time series"
#| label: fig-theta-decomp-fc
#| fig-pos: 'ht'

theta_forecasts = pd.DataFrame(
    {
        "original": sea2,
        "theta forecast": theta_mod_fit.forecast(24)
    }
)
theta_forecasts.tail(360).plot(figsize=(12,4));

### Forecasting with Seasonal Decomposition {#sec-06-fc-decomp}
### Example: Tourism forecasts with decomposition

In [ ]:
#| fig-align: center
#| fig-cap: Forecasting with STL decomposition
#| label: fig-decomp-fc
#| fig-pos: 'ht'

stlf = STLForecast(sea2, ARIMA, model_kwargs={"order": (3, 1, 2)})
res = stlf.fit( )
forecasts2 = pd.DataFrame(
    {
        "original": sea2,
        "dcmp forecast": res.forecast(24)
    }
)
forecasts2.tail(360).plot(figsize=(12,4));

## Miscellaneous Topics
### Time Series Clustering
### Example: Time series clustering, US employment data

In [ ]:
us_employment = pd.read_csv("data/us_employment.csv", parse_dates=[0], 
                            date_format="%Y %b")
series_ids = us_employment.Series_ID.unique()
us_employment.sample(n=8)

In [ ]:
#| fig-align: center
#| fig-cap: Sample time series from US employment time series
#| label: fig-emp-ts-example
#| fig-pos: 'ht'

us_employment[us_employment.Title == "Total Private"].plot(x='Month', 
                                                           y='Employed', 
                                                           figsize=(12,4));

In [ ]:
us_full = us_employment[(us_employment.Month >= datetime.datetime(2002, 9, 30)) & (us_employment.Month < datetime.datetime(2018, 1, 1) )   ]
us2 = us_full.pivot(index='Series_ID', columns='Month', values="Employed")
us2_array = us2.to_numpy()

In [ ]:
#| fig-align: center
#| fig-cap: Dendrogram for employment time series clustering
#| label: fig-emp-dendrogram
#| fig-pos: 'ht'

out = pdist(us2_array, metric='correlation')
lm1 = hierarchy.linkage(out, method='average', optimal_ordering=True)

plt.figure(figsize=(12,5))
hierarchy.dendrogram(lm1, p=3, truncate_mode='level',color_threshold=True);

In [ ]:
#| fig-align: center
#| label: fig-heatmap-correl
#| fig-cap: "Heatmap of correlation matrix"
#| fig-pos: 'ht'
 
X_ord = us2_array[hierarchy.leaves_list(lm1)]
corr_mat_ord = np.corrcoef(X_ord)
plt.figure(figsize=(15, 15))
sns.heatmap(corr_mat_ord, vmin=-1, vmax=1, cmap='coolwarm_r', center=0);

In [ ]:
#| fig-align: center
#| label: fig-sim-1
#| fig-cap: Series from top left corner of heatmap in @fig-heatmap-correl
#| fig-pos: 'ht'

us2_series = us2.index.to_list()
ss = [us2_series[x] for x in [22, 24, 32, 33, 34]]
us2.T.loc[:, ss].plot(figsize=(12,4));
plt.legend(loc='upper right');

In [ ]:
#| fig-align: center
#| label: fig-sim-2
#| fig-cap: Series from bottom right corner of heatmap in @fig-heatmap-correl
#| fig-pos: 'ht'

ss = [us2_series[x] for x in [143, 144, 145]]
us2.T.loc[:, ss].plot(figsize=(12,4));

## Summary 
## References
### Stats models pages
### Forecasting principles and practice
## Exercises